In [ ]:
# Install libraries. We use --upgrade to avoid conflicts with Colab's pre-installed packages.
!pip install --upgrade pip
!pip install lamini datasets
!pip install torch transformers accelerate --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
INFO: pip is looking at multiple versions of lamini to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.9/894.9 kB 6.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 27.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.7 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [lamini]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires num

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 50.0 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 45.5 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 63.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 62.4 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 66.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 57.3 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 99.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 54.9 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 59.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 101.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 71.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.3 MB/s  0:00:00
  

In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

In [2]:
# 1. Load the Dataset (Customer Support Q&A)
data_path = "https://raw.githubusercontent.com/bitext/customer-support-llm-chatbot-training-dataset/main/data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
df = pd.read_csv(data_path)
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
df['intent'].value_counts()

,count
intent,
contact_customer_service,1000
complaint,1000
check_invoice,1000
switch_account,1000
edit_account,1000
contact_human_agent,999
check_payment_methods,999
delivery_period,999
newsletter_subscription,999


In [4]:
# 2. Filter for a specific intent
df = df.loc[df['intent']=='cancel_order']
df.shape

(998, 5)

In [5]:
# 3. Format it in a promp template (Question, Answer -> Instruction, Response)
training_data = []
for i in range(len(df)):
  entry = {
      'text' : f"### Question:\n{df.iloc[i]['instruction']}\n\n ### Answer:\n{df.iloc[i]['response']}"
  }
  training_data.append(entry)

In [6]:
training_data[:1]

[{'text': "### Question:\nquestion about cancelling order {{Order Number}}\n\n ### Answer:\nI've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}]

In [7]:
len(training_data)

998

In [8]:
# Convert it this into a hugging face dataset
dataset = Dataset.from_pandas(pd.DataFrame(training_data))
dataset

Dataset({
    features: ['text'],
    num_rows: 998
})

In [9]:
# 4. Load Tokenizer (small and fast model for demo)
model_name = "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [10]:
# Tokenization Function :
def tokenize_function(examples) :
  return tokenizer(
      examples['text'],
      padding = 'max_length',
      truncation = True,
      max_length = 512
  )

In [11]:
tokenized_datasets = dataset.map(tokenize_function,batched = True)

Map:   0%|          | 0/998 [00:00<?, ? examples/s]

In [12]:
tokenized_datasets

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 998
})

In [13]:
tokenized_datasets = tokenized_datasets.add_column("labels",tokenized_datasets['input_ids'])

In [14]:
# Split the data in train, test
split_datasets = tokenized_datasets.train_test_split(test_size=0.1,seed=42)

In [15]:
split_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 898
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [16]:
training_dataset = split_datasets['train']
test_dataset     = split_datasets['test']

In [17]:
# attention mask :
# input ids [101,102,103,,0,0]
# attention mask [1,1,1,,0,0]
# use real tokens and ignore the padding tokens

In [18]:
import torch

In [19]:
# 1. Load the base model
base_model = AutoModelForCausalLM.from_pretrained(model_name)


# 2. Define Training arguments
training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 1, # total number of epochs to be performed
    per_device_train_batch_size = 2, # batch size per device during training
    learning_rate = 2e-5,
    logging_steps = 10,
    optim = 'adamw_torch',
    save_strategy = 'no') # Do not save checkpoints during training


# 3. Initialize the trainer
trainer = Trainer(
    model = base_model,
    args = training_args,
    train_dataset = training_dataset,
    eval_dataset = test_dataset
)


# 4. Start the training
trainer.train()

model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Step,Training Loss
10,3.397760
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


TrainOutput(global_step=449, training_loss=0.07567394231103843, metrics={'train_runtime': 20.9149, 'train_samples_per_second': 42.936, 'train_steps_per_second': 21.468, 'total_flos': 123231855968256.0, 'train_loss': 0.07567394231103843, 'epoch': 1.0})

In [20]:
# --- INFERENCE (TESTING) ---

# Function to generate text from the model
def run_inference(question, model):
    # Format the input just like training data
    prompt = f"### Question:\n{question}\n\n### Answer:\n"

    # Encode input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode back to text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [21]:
# Test the model
test_question = "I want to cancel my order please."
print("\n--- Model Output ---")
print(run_inference(test_question, base_model))


--- Model Output ---
### Question:
I want to cancel my order please.

### Answer:



In [22]:
# TinyLlama!

In [23]:
# --- 1. SETUP DATA ---
data_path = "https://raw.githubusercontent.com/bitext/customer-support-llm-chatbot-training-dataset/main/data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
df = pd.read_csv(data_path)
# Filter for 'cancel_order' intent, as done for the previous model, to ensure consistent data context
# df = df.loc[df['intent']=='cancel_order']
# df = df.head(5000)
df = df[df['intent'].isin(['cancel_order','complaint'])]

# Format Data
training_data = []
for i in range(len(df)):
    entry = {'text': f"### Question:\n{df.iloc[i]['instruction']}\n\n### Answer:\n{df.iloc[i]['response']}"}
    training_data.append(entry)

dataset = Dataset.from_pandas(pd.DataFrame(training_data))

In [24]:
# 2. Load model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [26]:
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                  torch_dtype = torch.float16,
                                                  device_map = "auto")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [27]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512) # Increased max_length

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.add_column("labels", tokenized_datasets["input_ids"])
split_dataset = tokenized_datasets.train_test_split(test_size=0.1, seed=42)

# --- 3. TRAIN (Safe Settings) ---
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,  # Safe batch size
    gradient_accumulation_steps=4,
    learning_rate=2e-5,             # Low learning rate to prevent "gibberish" collapse
    logging_steps=10,
    optim="adamw_torch",
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
)

Map:   0%|          | 0/1998 [00:00<?, ? examples/s]

In [28]:
trainer.train()

Step,Training Loss
10,1132.412109
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


TrainOutput(global_step=450, training_loss=25.164713541666668, metrics={'train_runtime': 448.3125, 'train_samples_per_second': 4.011, 'train_steps_per_second': 1.004, 'total_flos': 5714083634479104.0, 'train_loss': 25.164713541666668, 'epoch': 1.0})

In [29]:
print("\n--- Output ---")
print(run_inference("I want to know about cancellation.", base_model))


--- Output ---
### Question:
I want to know about cancellation.

### Answer:

